Test functions for the 4 edge cases: orphan order_items, discount > 100, zero quantity, future-dated orders.

In [0]:
import pandas as pd
from datetime import datetime, timedelta

BASE_DIR="/Volumes/assignment8/assignment8schema/assignment8volume"

order_items_df=pd.read_csv(f"{BASE_DIR}/order_items.csv")
orders_df=pd.read_csv(f"{BASE_DIR}/orders.csv")
orders_clean=pd.read_csv(f"{BASE_DIR}/orders_clean.csv")

def check_referential_integrity(order_items_df, orders_df):
    valid_ids=set(orders_df["order_id"])
    bad_rows=order_items_df[~order_items_df["order_id"].isin(valid_ids)]
    return bad_rows

In [0]:
def test_order_items_orphan_reference():
    orphans=check_referential_integrity(order_items_df, orders_df)
    assert len(orphans)>=1, "Expected at least one orphaned order_item"
    assert not orphans.empty
    print(f"PASS: found {len(orphans)} orphaned order_items (order_id not in orders).")

def test_discount_over_100():
    bad=order_items_df[order_items_df["discount_percent"]>100]
    assert len(bad)>=1, "Expected at least one row with discount_percent > 100"
    print(f"PASS: found {len(bad)} row(s) with discount_percent > 100.")

def test_zero_quantity():
    zero_qty=order_items_df[order_items_df["quantity"]==0]
    assert len(zero_qty)>=1, "Expected at least one row with quantity == 0"
    revenue_contribution=(
        zero_qty["quantity"]*zero_qty["unit_price"]*(1-zero_qty["discount_percent"]/100.0)
    ).sum()
    assert revenue_contribution==0
    print(f"PASS: found {len(zero_qty)} row(s) with quantity == 0; revenue contribution is {revenue_contribution}.")

def test_future_dated_order():
    fake_future_order=pd.DataFrame([{
        "order_id": 999999,
        "customer_id": 1,
        "order_date": (datetime.now()+timedelta(days=30)).strftime("%Y-%m-%d %H:%M:%S"),
        "status": "PLACED",
        "region_code": "NORTH",
    }])
    combined=pd.concat([orders_clean, fake_future_order], ignore_index=True)
    future_orders=combined[pd.to_datetime(combined["order_date"])>pd.Timestamp.now()]
    assert len(future_orders)>=1
    print(f"PASS: detected {len(future_orders)} future-dated order(s).")

In [0]:
for test_fn in [
    test_order_items_orphan_reference,
    test_discount_over_100,
    test_zero_quantity,
    test_future_dated_order,
]:
    test_fn()

PASS: found 1 orphaned order_items (order_id not in orders).
PASS: found 1 row(s) with discount_percent > 100.
PASS: found 1 row(s) with quantity == 0; revenue contribution is 0.0.
PASS: detected 1 future-dated order(s).
